## TUGAS MANDIRI - 2320506038 - Debi Aprilia
### Konteks / Skenario

Manajemen platform e-commerce meminta dibuatkan **dashboard performa cabang toko** yang menggabungkan data transaksi (yang sudah ada di HDFS sejak Pertemuan 3-4) dengan data referensi target penjualan tiap cabang. Anda ditugaskan menyiapkan analisis ini menggunakan kombinasi **join, window function, dan Spark SQL** — persis seperti yang dipelajari hari ini.

### Menyiapkan Dataset

Jalankan cell berikut untuk membuat **dua tabel** dan mengunggah tabel transaksi ke HDFS (tabel target cukup dibuat langsung sebagai Spark DataFrame, karena berukuran kecil dan jarang berubah — praktik umum untuk tabel referensi/*dimension table*).

In [1]:
import numpy as np
import pandas as pd

# Tabel 1: Target & PIC per cabang kota (tabel referensi, dibuat langsung sebagai DataFrame)
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

# Tabel 2: Data transaksi (disimpan sebagai CSV, lalu diunggah ke HDFS)
np.random.seed(55)
n = 500
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
}
pd.DataFrame(data_transaksi_t5).to_csv("transaksi_tugas5.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas5
!hdfs dfs -put -f transaksi_tugas5.csv /user/mahasiswa/tugas5/
print("Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/mahasiswa/tugas5/transaksi_tugas5.csv")
print("Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.")

Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/mahasiswa/tugas5/transaksi_tugas5.csv
Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.


### Instruksi Pengerjaan

Buat notebook baru **`Tugas5_[NPM]_[Nama Lengkap].ipynb`**, buat `SparkSession`, lalu:
1. Baca `transaksi_tugas5.csv` **dari HDFS** menjadi `df_transaksi`, tambahkan kolom `pendapatan` (`unit_terjual x harga_satuan`).
2. Buat `df_target` dari dictionary `data_target_cabang` di atas

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, count, row_number
from pyspark.sql.window import Window

#Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("TugasMandiri5") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

#Membaca data transaksi dari HDFS dan menambahkan kolom pendapatan
df_transaksi = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas5/transaksi_tugas5.csv",
    header=True,
    inferSchema=True
)
df_transaksi = df_transaksi.withColumn("pendapatan", col("unit_terjual") * col("harga_satuan"))

#Membuat df_target dari dictionary data_target_cabang
df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))

#Menampilkan data awal
df_transaksi.show(5)
df_target.show()

+--------+--------------------+----------+------------+------------+----------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|pendapatan|
+--------+--------------------+----------+------------+------------+----------+
|   TRX-0|   Makanan & Minuman| Purworejo|           8|       75000|    600000|
|   TRX-1|          Elektronik|      Solo|           9|       75000|    675000|
|   TRX-2|Kesehatan & Kecan...|      Solo|           6|      100000|    600000|
|   TRX-3|             Fashion|Yogyakarta|           6|      100000|    600000|
|   TRX-4|          Elektronik|Yogyakarta|           2|       75000|    150000|
+--------+--------------------+----------+------------+------------+----------+
only showing top 5 rows



+----------+--------------+----------+
|      kota|target_bulanan|pic_cabang|
+----------+--------------+----------+
|  Magelang|      45000000|      Rani|
|Yogyakarta|      60000000|      Joko|
|  Semarang|      55000000|      Sari|
|      Solo|      40000000|      Bayu|
| Purworejo|      30000000|     Fitri|
+----------+--------------+----------+



**A. Join & Perbandingan Target** *(bobot 25%)*

Ringkas total `pendapatan` per `kota` dari `df_transaksi`, lalu **join** dengan `df_target`. Tambahkan kolom `pencapaian_persen`. Urutkan hasil dari pencapaian tertinggi.

In [5]:
#Meringkas total pendapatan per kota dari df_transaksi
df_ringkasan_kota = df_transaksi.groupBy("kota") \
    .agg(spark_sum("pendapatan").alias("total_pendapatan"))

#Melakukan join dengan df_target dan menambahkan kolom pencapaian_persen
df_hasil_a = df_ringkasan_kota.join(df_target, on="kota", how="inner") \
    .withColumn("pencapaian_persen", (col("total_pendapatan") / col("target_bulanan")) * 100) \
    .orderBy(col("pencapaian_persen").desc())

#Menampilkan hasil
df_hasil_a.show()

[Stage 8:>                                                          (0 + 2) / 2]

+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|        45650000|      30000000|     Fitri|152.16666666666669|
|      Solo|        33475000|      40000000|      Bayu|           83.6875|
|Yogyakarta|        47275000|      60000000|      Joko| 78.79166666666667|
|  Magelang|        31650000|      45000000|      Rani| 70.33333333333334|
|  Semarang|        38175000|      55000000|      Sari|  69.4090909090909|
+----------+----------------+--------------+----------+------------------+



**B. Window Function — Kategori Terlaris per Kota** *(bobot 25%)*

Menggunakan window function, tentukan **kategori dengan pendapatan tertinggi di setiap kota** (top-1 saja, gunakan `row_number()`).

In [6]:
#Melakukan agregasi total pendapatan per kota dan per kategori
df_pendapatan_kategori = df_transaksi.groupBy("kota", "kategori") \
    .agg(spark_sum("pendapatan").alias("total_pendapatan_kategori"))

#Melakukan partisi berdasarkan kota dan urutan pendapatan terbesar
window_kota = Window.partitionBy("kota").orderBy(col("total_pendapatan_kategori").desc())

#Memberi nomor urut menggunakan row_number() dan mengambil hanya peringkat 1
df_hasil_b = df_pendapatan_kategori.withColumn("rn", row_number().over(window_kota)) \
    .filter(col("rn") == 1) \
    .drop("rn") \
    .orderBy("kota")

#Menampilkan Hasil
df_hasil_b.show()

[Stage 11:>                                                         (0 + 1) / 1]

+----------+--------------------+-------------------------+
|      kota|            kategori|total_pendapatan_kategori|
+----------+--------------------+-------------------------+
|  Magelang|Kesehatan & Kecan...|                  7275000|
| Purworejo|Kesehatan & Kecan...|                 10075000|
|  Semarang|        Rumah Tangga|                 11125000|
|      Solo|Kesehatan & Kecan...|                  8425000|
|Yogyakarta|             Fashion|                 13325000|
+----------+--------------------+-------------------------+



**C. Spark SQL** *(bobot 25%)*

Daftarkan `df_transaksi` dan `df_target` sebagai *temporary view*, lalu **tulis satu kueri SQL** (bukan DataFrame API) yang menampilkan: `kota`, `pic_cabang`, dan jumlah transaksi (`COUNT`) di kota tersebut, diurutkan dari jumlah transaksi terbanyak.

In [7]:
#Mendaftarkan df_transaksi dan df_target sebagai temporary view
df_transaksi.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("target")

#Menuliskan kueri SQL
df_hasil_c = spark.sql('''
    SELECT t.kota, tg.pic_cabang, COUNT(t.order_id) AS jumlah_transaksi
    FROM transaksi t
    JOIN target tg ON t.kota = tg.kota
    GROUP BY t.kota, tg.pic_cabang
    ORDER BY jumlah_transaksi DESC
''')

#Menampilkan hasil
df_hasil_c.show()

[Stage 16:=============================>                            (1 + 1) / 2]

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



**D. Kesimpulan** *(bobot 25%)*

Tulis pada markdown cell (**minimal 100 kata**): berdasarkan hasil bagian A dan B, **cabang mana yang berkinerja paling baik** dan **cabang mana yang paling perlu perhatian manajemen**? Sertakan angka-angka pendukung dari hasil analisis kalian, bukan opini tanpa dasar data.

**Jawaban :**
Berdasarkan hasil analisis pada bagian A dan B, cabang yang memiliki kinerja paling baik adalah cabang **Purworejo** dibawah kepemimpinan Fitri dengan persentase pencapaian tertinggi yaitu sebesar 152,16%, dengan total pendapatan 45.650.000 yang sudah melebihi target bulanannya yaitu sebesar 30.000.000. Cabang Purworejo menjadi satu - satunya cabang yang berhasil mencapai bahkan melebihi target bulanannya. Kategori yang menyumbang pendapatan terbesar di cabang Purworejo adalah kategori Kesehatan dan Kecantikan dengan total pendapatan kategori sebesar 10.075.000. \
Sedangkan, cabang yang membutuhkan perhatian manajemen adalah **Semarang** dan **Magelang**. Cabang **Semarang** memiliki persentase pencapaian terendah yaitu sebesar 69,40%, dengan total pendapatan 38.175.000 yang mana pendapatan tersebut jauh dari target bulanannya yaitu sebesar 55.000.000. Kategori Rumah Tangga di cabang Semarang yang menjadi kategori dengan pendapatan terbesar di cabang tersebut yaitu sebesar 11.125.000 masih belum dapat mengejar ketertinggalan target bulanannya. Cabang **Magelang** juga membutuhkan perhatian karena memiliki total pendapatan terkecil dari cabang lain yaitu sebesar 31.650.000 dari target 45.000.000. Cabang Magelang memiliki persentase pencapaian terendah kedua yaitu sebesar 70,33%. Dibandingkan dengan kota lain dari kategori yang menghasilkan pendapatan terbesar di setiap cabang, Magelang hanya menghasilkan 7.275.000. Pendapatan terbesar di cabang Magelang pada kategori Kesehatan dan Kecantikan ini merupakan pendapatan kategori dengan angka terkecil dibanding kota lainnya. Pihak manajemen diharapkan dapat melakukan evaluasi terhadap strategi pemasaran berdasarkan kategori potensial di kedua cabang tersebut.

## Tutup SparkSession

In [8]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
